In [ ]:
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error

if xgb.__version__ != "3.2.0":
    raise RuntimeError(f"B05 parity requires XGBoost 3.2.0; found {xgb.__version__}")

_cwd = Path.cwd().resolve()
ROOT = next((p for p in [_cwd, *_cwd.parents]
             if (p / "data/processed/split_trip_ids.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Could not locate SupplyShield project root")
DATA = ROOT / "data/processed"
EXP_DAY2 = ROOT / "experiments/day2/master_split"
EXP_DAY3 = ROOT / "experiments/day3/master_split_v2"
DAY5 = ROOT / "experiments/day5"
RISK_PATH = DAY5 / "risk_table_master_split_v1.csv"
MANIFEST_PATH = DAY5 / "risk_table_master_split_v1_manifest.json"
SUCCESS_PATH = DAY5 / "risk_table_master_split_v1_SUCCESS.json"


In [ ]:
# Load only the authoritative master-split/core inputs and current B02/B03 artifacts.
split_ids = pd.read_csv(DATA / "split_trip_ids.csv", parse_dates=["trip_creation_time"])
legs = pd.read_csv(DATA / "legs.csv", parse_dates=["od_start_time", "trip_creation_time"])
legs["od_end_time"] = pd.to_datetime(legs["od_end_time"], format="mixed")
core_train = pd.read_csv(DATA / "core_train_legs.csv", parse_dates=["od_start_time", "trip_creation_time"])
core_train["od_end_time"] = pd.to_datetime(core_train["od_end_time"], format="mixed")
core_edges = pd.read_csv(DATA / "core_edges.csv")

MODEL_PATH = ROOT / "models/residual_xgb_plain_master_split_v1.json"
CARD_PATH = EXP_DAY2 / "final_model_card_master_split_v1.json"
B02_PRED_PATH = DATA / "preds_cal_test_master_split_v1.csv"
PARAMS_PATH = EXP_DAY3 / "final/conformal_params_method_c_master_split_v2.json"
B03_MANIFEST_PATH = EXP_DAY3 / "run_manifest.json"
B03_SUCCESS_PATH = EXP_DAY3 / "_SUCCESS.json"
B03_TEST_PATH = EXP_DAY3 / "final/test_intervals_method_c_master_split_v2.csv"
B03_METRICS_PATH = EXP_DAY3 / "evaluation/interval_comparison_all_methods.csv"

for required in [MODEL_PATH, CARD_PATH, B02_PRED_PATH, PARAMS_PATH, B03_MANIFEST_PATH,
                 B03_SUCCESS_PATH, B03_TEST_PATH, B03_METRICS_PATH]:
    if not required.is_file():
        raise FileNotFoundError(required)

model_card = json.loads(CARD_PATH.read_text(encoding="utf-8"))
conformal_params = json.loads(PARAMS_PATH.read_text(encoding="utf-8"))
b03_manifest = json.loads(B03_MANIFEST_PATH.read_text(encoding="utf-8"))
b03_success = json.loads(B03_SUCCESS_PATH.read_text(encoding="utf-8"))
b02_saved = pd.read_csv(B02_PRED_PATH, parse_dates=["od_start_time"])
b03_test_saved = pd.read_csv(B03_TEST_PATH, parse_dates=["od_start_time"])
b03_metrics = pd.read_csv(B03_METRICS_PATH)

assert b03_manifest["status"] == b03_success["status"] == "complete"
assert conformal_params["experiment"] == "day3_master_split_v2"
assert conformal_params["input_prediction_file"].replace("\\", "/") == "data/processed/preds_cal_test_master_split_v1.csv"
assert b03_manifest["input"].replace("\\", "/") == "data/processed/preds_cal_test_master_split_v1.csv"
assert model_card["split_source"].replace("\\", "/") == "data/processed/split_trip_ids.csv"
assert not any((DAY5 / n).exists() for n in [RISK_PATH.name, MANIFEST_PATH.name, SUCCESS_PATH.name]), \
    "Refusing to overwrite an existing B05 output; choose a new versioned name"


In [ ]:
# Reconstruct B02's exact master-split as-of feature pipeline and verify the saved predictions.
FEATURES = model_card["features"]
EXPECTED_FEATURES = [
    "osrm_time_min", "osrm_distance_km", "osrm_speed_kmh", "log_osrm_time",
    "start_hour", "start_dow", "is_weekend", "is_ftl", "lane_base", "lane_history_count"
]
assert FEATURES == EXPECTED_FEATURES

split_map = split_ids.set_index("trip_uuid")["split"]
assert split_ids["trip_uuid"].is_unique
assert set(split_ids["split"].unique()) == {"train", "calibration", "test"}
legs["split"] = legs["trip_uuid"].map(split_map)
assert legs["split"].notna().all()
train = legs[legs["split"] == "train"].copy()
cal = legs[legs["split"] == "calibration"].copy()
test = legs[legs["split"] == "test"].copy()

train_trip_dates = train.groupby("trip_uuid")["trip_creation_time"].min().sort_values()
fit_trip_ids = train_trip_dates.index[:int(len(train_trip_dates) * 0.8)]
fit_df = train[train["trip_uuid"].isin(fit_trip_ids)].copy()
g_fit = float(fit_df["delay_ratio"].median())

def add_historical_lane_features_asof(target_df, history_df):
    result = target_df.copy()
    lane_cols = ["source_center", "destination_center"]
    result["lane_median_ratio"] = 1.0
    result["lane_history_count"] = 0
    history_groups = {
        key: group.sort_values("od_end_time")
        for key, group in history_df.groupby(lane_cols, sort=False)
    }
    for key, target_group in result.groupby(lane_cols, sort=False):
        if key not in history_groups:
            continue
        hist = history_groups[key]
        end_times = hist["od_end_time"].to_numpy(dtype="datetime64[ns]")
        delays = hist["delay_ratio"].to_numpy()
        for idx, row in target_group.iterrows():
            start_time = np.datetime64(row["od_start_time"], "ns")
            count = np.searchsorted(end_times, start_time, side="right")
            result.loc[idx, "lane_history_count"] = count
            if count > 0:
                result.loc[idx, "lane_median_ratio"] = np.median(delays[:count])
    return result

def build_features(frame):
    start = pd.to_datetime(frame["od_start_time"])
    X = pd.DataFrame(index=frame.index)
    X["osrm_time_min"] = frame["osrm_time_min"]
    X["osrm_distance_km"] = frame["osrm_distance_km"]
    X["osrm_speed_kmh"] = frame["osrm_distance_km"] / (frame["osrm_time_min"] / 60)
    X["log_osrm_time"] = np.log1p(frame["osrm_time_min"])
    X["start_hour"] = start.dt.hour
    X["start_dow"] = start.dt.dayofweek
    X["is_weekend"] = (start.dt.dayofweek >= 5).astype(int)
    X["is_ftl"] = (frame["route_type"] == "FTL").astype(int)
    return X

def base_pred(frame):
    return frame["lane_median_ratio"].where(frame["lane_history_count"] > 0, g_fit).to_numpy()

def make_X(frame):
    X = build_features(frame)[FEATURES[:8]].copy()
    X["lane_base"] = base_pred(frame)
    X["lane_history_count"] = frame["lane_history_count"].to_numpy()
    return X[FEATURES]

cal_v2 = add_historical_lane_features_asof(cal, train)
test_history = pd.concat([train, cal], ignore_index=True)
test_v2 = add_historical_lane_features_asof(test, test_history)
model = xgb.XGBRegressor()
model.load_model(str(MODEL_PATH))
assert model.best_iteration == int(model_card["best_iteration"]) == 134
assert model.get_booster().feature_names == FEATURES
assert model.get_booster().num_boosted_rounds() == 235

reproduced_parts = []
for pile, frame in [("cal", cal_v2), ("test", test_v2)]:
    X = make_X(frame)
    assert list(X.columns) == FEATURES
    residual = model.predict(X)  # B02 cell 83: sklearn wrapper default uses saved best_iteration.
    base = base_pred(frame)
    reproduced_parts.append(pd.DataFrame({
        "trip_uuid": frame["trip_uuid"].to_numpy(),
        "source_center": frame["source_center"].to_numpy(),
        "destination_center": frame["destination_center"].to_numpy(),
        "pile": pile,
        "base_pred_reproduced": base,
        "residual_reproduced": residual,
        "pred_ratio_reproduced": base + residual,
        "lane_history_count_reproduced": frame["lane_history_count"].to_numpy(),
    }))

b02_reproduced = pd.concat(reproduced_parts, ignore_index=True)
PRED_KEYS = ["trip_uuid", "source_center", "destination_center", "pile"]
b02_join = b02_reproduced.merge(b02_saved, on=PRED_KEYS, how="outer", validate="one_to_one", indicator=True)
assert len(b02_join) == 8695 and (b02_join["_merge"] == "both").all()
assert np.array_equal(b02_join["lane_history_count_reproduced"], b02_join["lane_history_count"])
BASE_MAX_DIFF = float(np.max(np.abs(b02_join["base_pred_reproduced"] - b02_join["base_pred"])))
RESIDUAL_MAX_DIFF = float(np.max(np.abs(b02_join["residual_reproduced"] - (b02_join["pred_ratio"] - b02_join["base_pred"]))))
B02_MAX_DIFF = float(np.max(np.abs(b02_join["pred_ratio_reproduced"] - b02_join["pred_ratio"])))
assert BASE_MAX_DIFF <= 1e-6 and RESIDUAL_MAX_DIFF <= 1e-6 and B02_MAX_DIFF <= 1e-6

B02_METRICS = {}
for pile, label in [("cal", "calibration"), ("test", "test")]:
    part = b02_saved[b02_saved["pile"] == pile]
    repro = b02_join[b02_join["pile"] == pile]
    mae = float(mean_absolute_error(part["y_ratio"], repro["pred_ratio_reproduced"]))
    saved_mae = float(mean_absolute_error(part["y_ratio"], part["pred_ratio"]))
    card_mae = float(model_card["results"][f"{label} - residual XGBoost"]["MAE_ratio"])
    assert abs(mae - saved_mae) <= 1e-12 and abs(mae - card_mae) <= 0.00005
    B02_METRICS[pile] = {"rows": int(len(part)), "mae_ratio": mae, "saved_mae_ratio": saved_mae, "card_mae_ratio": card_mae}


In [ ]:
# Rebuild Method C intervals from saved B03 parameters and verify every saved test interval.
cal_pred = b02_saved[b02_saved["pile"] == "cal"].copy()
test_pred = b02_saved[b02_saved["pile"] == "test"].copy()
DISTANCE_EDGES = [float(x) for x in conformal_params["distance_edges_km"]]
assert np.allclose(DISTANCE_EDGES, [31.21256666666666, 60.86566666666664], rtol=0, atol=1e-12)
cal_quantiles = cal_pred["osrm_distance_km"].quantile([1/3, 2/3]).to_numpy()
assert np.allclose(cal_quantiles, DISTANCE_EDGES, rtol=0, atol=1e-12)

def add_method_c_group(frame):
    out = frame.copy()
    out["dist_bucket"] = pd.cut(
        out["osrm_distance_km"], [-np.inf, DISTANCE_EDGES[0], DISTANCE_EDGES[1], np.inf],
        labels=["short", "medium", "long"], include_lowest=True
    ).astype("object")
    out["grp"] = out["route_type"].astype(str) + "|" + out["dist_bucket"].astype(str)
    return out

def apply_method_c(frame):
    out = frame.copy()
    for level in ["50%", "80%", "90%"]:
        info = conformal_params["levels"][level]
        offsets = [info["group_offsets"].get(g, info["global_offsets"]) for g in out["grp"]]
        lower_offset = np.array([x[0] for x in offsets], dtype=float)
        upper_offset = np.array([x[1] for x in offsets], dtype=float)
        tag = level.replace("%", "")
        out[f"lo{tag}"] = np.clip(out["pred_ratio"].to_numpy() + lower_offset, 0, None)
        out[f"hi{tag}"] = out["pred_ratio"].to_numpy() + upper_offset
    return out

test_method_c = apply_method_c(add_method_c_group(test_pred))
INTERVAL_KEYS = ["trip_uuid", "source_center", "destination_center", "od_start_time"]
b03_join = test_method_c.merge(b03_test_saved, on=INTERVAL_KEYS, how="outer", validate="one_to_one", indicator=True, suffixes=("_reproduced", "_saved"))
assert len(b03_join) == 4394 and (b03_join["_merge"] == "both").all()
assert (b03_join["grp_reproduced"] == b03_join["grp_saved"]).all()
assert (b03_join["dist_bucket_reproduced"] == b03_join["dist_bucket_saved"]).all()
assert np.allclose(b03_join["pred_ratio_reproduced"], b03_join["pred_ratio_saved"], rtol=0, atol=1e-12)
for tag in ["50", "80", "90"]:
    assert np.allclose(b03_join[f"lo{tag}"], b03_join[f"lower_{tag}"], rtol=0, atol=1e-12)
    assert np.allclose(b03_join[f"hi{tag}"], b03_join[f"upper_{tag}"], rtol=0, atol=1e-12)

B03_COVERAGE = {}
for tag, level in [("50", "50%"), ("80", "80%"), ("90", "90%")]:
    y = b03_join["y_ratio_reproduced"] if "y_ratio_reproduced" in b03_join else b03_join["y_ratio"]
    coverage = float(((y >= b03_join[f"lo{tag}"]) & (y <= b03_join[f"hi{tag}"])).mean())
    metric_row = b03_metrics[(b03_metrics["method"] == "C grouped (route x distance)") & (b03_metrics["level"] == level)]
    assert len(metric_row) == 1 and abs(coverage - float(metric_row.iloc[0]["coverage"])) <= 1e-12
    B03_COVERAGE[level] = {"test_rows": int(len(b03_join)), "test_trips": int(b03_join["trip_uuid"].nunique()), "coverage": coverage, "target": float(metric_row.iloc[0]["target"])}


In [ ]:
# Training-only representative lane-route rows at the common as-of cutoff.
CUTOFF = pd.Timestamp("2018-09-29 00:00:00")
LANE_KEYS = ["source_center", "destination_center"]
ROUTE_KEYS = LANE_KEYS + ["route_type"]
EDGE_KEYS = set(map(tuple, core_edges[LANE_KEYS].to_numpy()))
assert len(core_edges) == 292 and core_edges.duplicated(LANE_KEYS).sum() == 0

core_split = core_train["trip_uuid"].map(split_map)
assert core_split.notna().all() and set(core_split.unique()) == {"train"}
assert not core_train["delay_ratio"].isna().any()
assert not core_train["od_end_time"].isna().any()
core_pairs = list(map(tuple, core_train[LANE_KEYS].to_numpy()))
assert set(core_pairs) <= EDGE_KEYS

# Prove core_train_legs is exactly the master-training leg population on frozen directed lanes.
train_pairs = list(map(tuple, train[LANE_KEYS].to_numpy()))
train_core = train.loc[[pair in EDGE_KEYS for pair in train_pairs]].copy()
MATCH_COLS = ["trip_uuid", "source_center", "destination_center", "od_start_time", "od_end_time"]
assert len(train_core) == len(core_train)
left = train_core[MATCH_COLS].astype(str).sort_values(MATCH_COLS).reset_index(drop=True)
right = core_train[MATCH_COLS].astype(str).sort_values(MATCH_COLS).reset_index(drop=True)
assert left.equals(right), "core_train_legs differs from the master-training rows on frozen lanes"

eligible = core_train[core_train["od_end_time"] <= CUTOFF].copy()
assert len(eligible) == len(core_train), "Some core training history ends after the common cutoff"
assert set(eligible["trip_uuid"].map(split_map).unique()) == {"train"}
eligible["_start"] = pd.to_datetime(eligible["od_start_time"])
eligible["start_hour"] = eligible["_start"].dt.hour.astype(int)
eligible["start_dow"] = eligible["_start"].dt.dayofweek.astype(int)

# B02 history is lane-only, so lane_base/count aggregate all route types on a directed lane.
lane_history = (eligible.groupby(LANE_KEYS, sort=False)
                .agg(lane_median_ratio=("delay_ratio", "median"),
                     lane_history_count=("delay_ratio", "size"))
                .reset_index())

# Representative OSRM values and departure context are route-specific.
route_reps = (eligible.groupby(ROUTE_KEYS, sort=False)
              .agg(osrm_time_min=("osrm_time_min", "median"),
                   osrm_distance_km=("osrm_distance_km", "median"),
                   representative_training_legs=("trip_uuid", "size"))
              .reset_index())
context_counts = (eligible.groupby(ROUTE_KEYS + ["start_hour", "start_dow"], sort=False)
                  .size().rename("context_count").reset_index())
context_counts = context_counts.sort_values(
    ROUTE_KEYS + ["context_count", "start_hour", "start_dow"],
    ascending=[True, True, True, False, True, True], kind="mergesort"
)
modal_context = context_counts.drop_duplicates(ROUTE_KEYS, keep="first")[
    ROUTE_KEYS + ["start_hour", "start_dow", "context_count"]
]
representatives = route_reps.merge(modal_context, on=ROUTE_KEYS, how="left", validate="one_to_one")
representatives = representatives.merge(lane_history, on=LANE_KEYS, how="left", validate="many_to_one")
assert len(representatives) == 296 and representatives.duplicated(ROUTE_KEYS).sum() == 0
assert representatives["lane_history_count"].notna().all() and (representatives["lane_history_count"] > 0).all()
assert representatives["route_type"].notna().all()
representatives["is_weekend"] = (representatives["start_dow"] >= 5).astype(int)
representatives["is_ftl"] = (representatives["route_type"] == "FTL").astype(int)
representatives["osrm_speed_kmh"] = representatives["osrm_distance_km"] / (representatives["osrm_time_min"] / 60)
representatives["log_osrm_time"] = np.log1p(representatives["osrm_time_min"])
representatives["lane_base"] = representatives["lane_median_ratio"].where(
    representatives["lane_history_count"] > 0, g_fit
)
representatives["prediction_timestamp"] = CUTOFF.strftime("%Y-%m-%d %H:%M:%S")

assert set(map(tuple, representatives[LANE_KEYS].drop_duplicates().to_numpy())) == EDGE_KEYS
assert representatives["representative_training_legs"].sum() == len(eligible)
assert representatives["lane_history_count"].sum() >= len(eligible)


In [ ]:
# Score each representative leg with the current B02 sklearn model and apply Method C offsets.
X_lane = representatives[FEATURES].copy()
assert list(X_lane.columns) == model_card["features"] == model.get_booster().feature_names
assert np.isfinite(X_lane.to_numpy(dtype=float)).all()
residual_lane = model.predict(X_lane)  # default sklearn prediction uses best_iteration=134
lane_pred = representatives["lane_base"].to_numpy() + residual_lane

lane_scored = add_method_c_group(representatives)
lane_scored["residual_prediction"] = residual_lane
lane_scored["pred_ratio"] = lane_pred
lane_scored = apply_method_c(lane_scored)
lane_scored["osrm_time_h"] = lane_scored["osrm_time_min"] / 60
lane_scored["pred_time_h"] = lane_scored["pred_ratio"] * lane_scored["osrm_time_h"]
lane_scored["pred_excess_h"] = (lane_scored["pred_ratio"] - 1) * lane_scored["osrm_time_h"]
for tag in ["50", "80", "90"]:
    lane_scored[f"excess_lo{tag}_h"] = (lane_scored[f"lo{tag}"] - 1) * lane_scored["osrm_time_h"]
    lane_scored[f"excess_hi{tag}_h"] = (lane_scored[f"hi{tag}"] - 1) * lane_scored["osrm_time_h"]

OUTPUT_COLUMNS = [
    "source_center", "destination_center", "route_type", "prediction_timestamp",
    "dist_bucket", "grp", "representative_training_legs", "context_count",
    "lane_history_count", "lane_base", "start_hour", "start_dow", "is_weekend", "is_ftl",
    "osrm_time_min", "osrm_time_h", "osrm_distance_km", "osrm_speed_kmh", "log_osrm_time",
    "residual_prediction", "pred_ratio",
    "lo50", "hi50", "lo80", "hi80", "lo90", "hi90",
    "pred_time_h", "pred_excess_h",
    "excess_lo50_h", "excess_hi50_h", "excess_lo80_h", "excess_hi80_h", "excess_lo90_h", "excess_hi90_h"
]
risk_table = lane_scored[OUTPUT_COLUMNS].copy()


In [ ]:
# Validate representative lane-route scores independently of held-out B02/B03 evaluation.
assert len(risk_table) == 296
assert risk_table.duplicated(ROUTE_KEYS).sum() == 0
assert risk_table[ROUTE_KEYS].notna().all().all()
assert set(map(tuple, risk_table[LANE_KEYS].drop_duplicates().to_numpy())) == EDGE_KEYS
assert risk_table["route_type"].nunique() >= 1
assert risk_table["prediction_timestamp"].eq("2018-09-29 00:00:00").all()
assert (eligible["od_end_time"] <= CUTOFF).all()
assert set(eligible["trip_uuid"].map(split_map).unique()) == {"train"}
assert np.isfinite(risk_table.select_dtypes(include=[np.number]).to_numpy(dtype=float)).all()
assert not risk_table.isna().any().any()
assert not any(c in risk_table.columns for c in ["trip_uuid", "y_ratio", "delay_ratio", "actual_time_min"])
assert list(X_lane.columns) == FEATURES
assert risk_table["lane_history_count"].min() >= 3
assert risk_table["dist_bucket"].isin(["short", "medium", "long"]).all()
assert (risk_table["lo50"] >= 0).all() and (risk_table["lo80"] >= 0).all() and (risk_table["lo90"] >= 0).all()
assert (risk_table["lo50"] <= risk_table["pred_ratio"]).all() and (risk_table["pred_ratio"] <= risk_table["hi50"]).all()
assert (risk_table["lo80"] <= risk_table["pred_ratio"]).all() and (risk_table["pred_ratio"] <= risk_table["hi80"]).all()
assert (risk_table["lo90"] <= risk_table["pred_ratio"]).all() and (risk_table["pred_ratio"] <= risk_table["hi90"]).all()
assert (risk_table["lo50"] >= risk_table["lo80"] - 1e-12).all()
assert (risk_table["lo80"] >= risk_table["lo90"] - 1e-12).all()
assert (risk_table["hi50"] <= risk_table["hi80"] + 1e-12).all()
assert (risk_table["hi80"] <= risk_table["hi90"] + 1e-12).all()

# In-memory CSV round-trip check; no output path is touched until this passes.
_roundtrip = pd.read_csv(StringIO(risk_table.to_csv(index=False)))
assert _roundtrip.shape == risk_table.shape
assert _roundtrip[ROUTE_KEYS].equals(risk_table[ROUTE_KEYS].reset_index(drop=True))
assert np.allclose(_roundtrip["pred_ratio"], risk_table["pred_ratio"], rtol=1e-12, atol=1e-12)
assert np.allclose(_roundtrip[["lo50", "hi50", "lo80", "hi80", "lo90", "hi90"]],
                   risk_table[["lo50", "hi50", "lo80", "hi80", "lo90", "hi90"]], rtol=1e-12, atol=1e-12)

print("PASS representative rows:", len(risk_table), "| directed lanes:", len(EDGE_KEYS),
      "| route variants:", len(risk_table) - len(EDGE_KEYS))
print("PASS B02 prediction parity: rows=8695 | max base diff=", BASE_MAX_DIFF,
      "| max residual diff=", RESIDUAL_MAX_DIFF, "| max ratio diff=", B02_MAX_DIFF)
print("PASS B03 Method C test interval parity: rows=4394 | coverage:", B03_COVERAGE)


In [ ]:
# Keep test evaluation separate from the representative lane-route score table.
print("B02 held-out evaluation (leg rows):", B02_METRICS)
print("B03 Method C held-out test coverage (leg rows, not lane coverage):", B03_COVERAGE)
print("Representative lane-route rows are model scores for modal training departure contexts.")
print("Their conformal bounds are representative-leg intervals; no lane-aggregate coverage is claimed.")


# Day 5 B05 — Master-split representative lane risk table

## Conclusion and interpretation

The validated artifact contains one row per **directed core lane and route type** (296 rows across 292 directed lanes). Each row is a representative leg score for that lane-route using the modal observed training departure hour/weekday and median route-specific OSRM time/distance. The B02 lane baseline and history count use all master-training observations on the directed lane that completed by the common cutoff, `2018-09-29 00:00:00`.

B02 residual XGBoost predictions are combined with the saved B03 Method C route-type × calibration-distance-tercile offsets. The reported intervals apply to a representative leg under that constructed context. They do **not** establish coverage for a lane mean, lane aggregate, or operational future distribution. Held-out B02/B03 metrics are reported separately from the lane table.

## Limitations and weather boundary

The representative OSRM values are medians within each lane-route group. `start_hour` and `start_dow` are the modal observed **joint** training context; ties select the smallest hour, then smallest weekday (Monday=0). Lane history is pooled across route types because that is B02's lane-key semantics, while representative OSRM values and departure context are route-specific. Four lanes therefore appear twice, once per observed route type. Distance grouping and reporting both use B02's `osrm_distance_km`; `core_edges.distance_km` is not substituted.

The conformal offsets were calibrated on individual held-out legs. Applying them to a representative feature row does not confer lane-level aggregate coverage. Weather cells below remain exploratory only: they are not inputs to this B02 model, not part of the validated risk table, and their in-sample results are not held-out evaluation.


In [ ]:
# Write-once outputs. All model, conformal, lineage, interval, and in-memory CSV checks passed above.
for output_path in [RISK_PATH, MANIFEST_PATH, SUCCESS_PATH]:
    if output_path.exists():
        raise FileExistsError(f"Refusing to overwrite existing output: {output_path}")

DAY5.mkdir(parents=False, exist_ok=True)
_csv_text = risk_table.to_csv(index=False)
with RISK_PATH.open("x", encoding="utf-8", newline="") as f:
    f.write(_csv_text)

# Read back the new artifact and validate its persisted keys and scores before publishing success.
_saved = pd.read_csv(RISK_PATH)
assert len(_saved) == 296 and _saved.duplicated(ROUTE_KEYS).sum() == 0
assert _saved[ROUTE_KEYS].notna().all().all()
assert np.allclose(_saved["pred_ratio"], risk_table["pred_ratio"], rtol=1e-12, atol=1e-12)
assert np.allclose(_saved[["lo50", "hi50", "lo80", "hi80", "lo90", "hi90"]],
                   risk_table[["lo50", "hi50", "lo80", "hi80", "lo90", "hi90"]], rtol=1e-12, atol=1e-12)

def sha256_file(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

manifest = {
    "status": "complete",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "artifact": str(RISK_PATH.relative_to(ROOT)).replace("\\", "/"),
    "artifact_sha256": sha256_file(RISK_PATH),
    "lineage": {
        "split": "data/processed/split_trip_ids.csv",
        "training_legs": "data/processed/core_train_legs.csv",
        "all_legs_crosscheck": "data/processed/legs.csv",
        "frozen_directed_lanes": "data/processed/core_edges.csv",
        "prediction_cutoff": "2018-09-29 00:00:00",
        "history_eligibility": "od_end_time <= prediction_timestamp",
        "history_partition": "master train only"
    },
    "model": {
        "file": str(MODEL_PATH.relative_to(ROOT)).replace("\\", "/"),
        "card": str(CARD_PATH.relative_to(ROOT)).replace("\\", "/"),
        "xgboost_version": xgb.__version__,
        "best_iteration": int(model.best_iteration),
        "prediction_api": "XGBRegressor.predict default (best iteration)",
        "features": FEATURES,
        "prediction_parity_rows": int(len(b02_join)),
        "base_pred_max_abs_diff": BASE_MAX_DIFF,
        "residual_max_abs_diff": RESIDUAL_MAX_DIFF,
        "pred_ratio_max_abs_diff": B02_MAX_DIFF,
        "heldout_metrics": B02_METRICS
    },
    "conformal": {
        "parameters": str(PARAMS_PATH.relative_to(ROOT)).replace("\\", "/"),
        "method": conformal_params["method"],
        "distance_field": "osrm_distance_km",
        "distance_edges_km": DISTANCE_EDGES,
        "test_interval_artifact": str(B03_TEST_PATH.relative_to(ROOT)).replace("\\", "/"),
        "test_interval_parity_rows": int(len(b03_join)),
        "heldout_test_coverage": B03_COVERAGE
    },
    "representative_rows": int(len(_saved)),
    "directed_lanes": int(len(EDGE_KEYS)),
    "lane_route_rows": int(len(_saved)),
    "weather_included": False,
    "interpretation": "Representative-leg point scores and Method C intervals; no lane-aggregate coverage claim."
}
with MANIFEST_PATH.open("x", encoding="utf-8", newline="") as f:
    json.dump(manifest, f, indent=2)

success = {
    "status": "complete",
    "manifest": str(MANIFEST_PATH.relative_to(ROOT)).replace("\\", "/"),
    "artifact": str(RISK_PATH.relative_to(ROOT)).replace("\\", "/"),
    "artifact_sha256": manifest["artifact_sha256"],
    "validated_rows": int(len(_saved))
}
with SUCCESS_PATH.open("x", encoding="utf-8", newline="") as f:
    json.dump(success, f, indent=2)
print("DAY 5 B05 STATUS: COMPLETE")
print("Risk table:", RISK_PATH.relative_to(ROOT), "rows:", len(_saved))
print("Manifest:", MANIFEST_PATH.relative_to(ROOT))
print("Success marker:", SUCCESS_PATH.relative_to(ROOT))


In [13]:
import pandas as pd
from pathlib import Path

# 1. Load core train legs and weather daily data
core_train_legs = pd.read_csv("../data/processed/core_train_legs.csv")
weather_daily = pd.read_csv("../data/processed/weather_daily.csv")

print("Core train legs shape:", core_train_legs.shape)
print("Weather daily shape:", weather_daily.shape)

# 2. Preview columns to verify join keys (e.g., hub/center and date)
print("Core train columns:", core_train_legs.columns.tolist()[:6])
print("Weather daily columns:", weather_daily.columns.tolist()[:6])

Core train legs shape: (5395, 15)
Weather daily shape: (2470, 11)
Core train columns: ['trip_uuid', 'source_center', 'destination_center', 'od_start_time', 'od_end_time', 'trip_creation_time']
Weather daily columns: ['date', 'rain_mm', 'precipitation_mm', 'temperature_mean_c', 'hub', 'requested_latitude']


In [14]:
# Convert timestamps to datetime format for merging
core_train_legs["date"] = pd.to_datetime(
    core_train_legs["trip_creation_time"]
).dt.strftime("%Y-%m-%d")
weather_daily["date"] = pd.to_datetime(weather_daily["date"]).dt.strftime(
    "%Y-%m-%d"
)

# Merge weather data onto core train legs using source hub and date
merged_train = pd.merge(
    core_train_legs,
    weather_daily,
    left_on=["source_center", "date"],
    right_on=["hub", "date"],
    how="left",
)

print("Merged core train legs with weather shape:", merged_train.shape)
print("Missing rain_mm values:", merged_train["rain_mm"].isna().sum())
display(merged_train[["trip_uuid", "source_center", "date", "rain_mm"]].head())

Merged core train legs with weather shape: (5395, 26)
Missing rain_mm values: 239


,trip_uuid,source_center,date,rain_mm
0,trip-153671041653548748,IND209304AAA,2018-09-12,1.3
1,trip-153671041653548748,IND462022AAA,2018-09-12,0.3
2,trip-153671043369099517,IND000000ACB,2018-09-12,0.1
3,trip-153671043369099517,IND562132AAA,2018-09-12,2.6
4,trip-153671046011330457,IND400072AAB,2018-09-12,0.6


In [19]:
import numpy as np
from sklearn.metrics import root_mean_squared_error

# Fill missing rain values with 0 (dry days)
merged_train["rain_mm"] = merged_train["rain_mm"].fillna(0)

# Check if target delay ratio column exists (or compute it)
# Let's inspect available columns to ensure we use the correct target name
print(
    "Target / delay columns available:",
    [c for c in merged_train.columns if "ratio" in c or "delay" in c],
)

Target / delay columns available: ['delay_ratio']


In [17]:
print(merged_train.columns.tolist())

['trip_uuid', 'source_center', 'destination_center', 'od_start_time', 'od_end_time', 'trip_creation_time', 'source_name', 'destination_name', 'route_type', 'actual_time_min', 'osrm_time_min', 'osrm_distance_km', 'scan_rows', 'delay_ratio', 'trip_created', 'date', 'rain_mm', 'precipitation_mm', 'temperature_mean_c', 'hub', 'requested_latitude', 'requested_longitude', 'grid_latitude', 'grid_longitude', 'weather_model', 'timezone']


In [18]:
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor

# Fill missing values for the features and target
merged_train["rain_mm"] = merged_train["rain_mm"].fillna(0)
y = merged_train["delay_ratio"].fillna(1.0)

# Define feature sets using the exact column names available
features_base = ["osrm_distance_km", "osrm_time_min"]
features_weather = ["osrm_distance_km", "osrm_time_min", "rain_mm"]

X_base = merged_train[features_base].fillna(0)
X_weather = merged_train[features_weather].fillna(0)

# Train baseline vs weather-augmented models
rf_base = RandomForestRegressor(n_estimators=50, random_state=42)
rf_weather = RandomForestRegressor(n_estimators=50, random_state=42)

rf_base.fit(X_base, y)
rf_weather.fit(X_weather, y)

pred_base = rf_base.predict(X_base)
pred_weather = rf_weather.predict(X_weather)

rmse_base = root_mean_squared_error(y, pred_base)
rmse_weather = root_mean_squared_error(y, pred_weather)

print(f"RMSE without weather: {rmse_base:.4f}")
print(f"RMSE with weather (rain_mm): {rmse_weather:.4f}")
print(
    f"Improvement: {((rmse_base - rmse_weather) / rmse_base) * 100:.2f}% drop in error"
)

RMSE without weather: 0.7014
RMSE with weather (rain_mm): 0.6475
Improvement: 7.69% drop in error


In [20]:
from sklearn.metrics import root_mean_squared_error
from sklearn.ensemble import RandomForestRegressor

# Prepare features (with and without rain)
features_base = ["distance_km", "median_actual_time"]
features_weather = ["distance_km", "median_actual_time", "rain_mm"]

X_base = merged_train[features_base].fillna(0)
X_weather = merged_train[features_weather].fillna(0)
y = merged_train["delay_ratio"].fillna(1.0)

# Quick train/eval split or out-of-fold comparison
# Let's train a quick Random Forest baseline to check RMSE impact
rf_base = RandomForestRegressor(n_estimators=50, random_state=42)
rf_weather = RandomForestRegressor(n_estimators=50, random_state=42)

rf_base.fit(X_base, y)
rf_weather.fit(X_weather, y)

pred_base = rf_base.predict(X_base)
pred_weather = rf_weather.predict(X_weather)

rmse_base = root_mean_squared_error(y, pred_base)
rmse_weather = root_mean_squared_error(y, pred_weather)

print(f"RMSE without weather: {rmse_base:.4f}")
print(f"RMSE with weather (rain_mm): {rmse_weather:.4f}")
print(
    f"Improvement: {((rmse_base - rmse_weather) / rmse_base) * 100:.2f}% drop in"
    " error"
)

KeyError: "None of [Index(['distance_km', 'median_actual_time'], dtype='object')] are in the [columns]"